In [ ]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline
from tqdm import tqdm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

val_data = load_jsonl("hellaswag_val.300.jsonl")
test_data = load_jsonl("hellaswag_test2.1k.student.jsonl")
train_data = load_jsonl("hellaswag_train.jsonl")

print(f"Val: {len(val_data)}, Test: {len(test_data)}, Train: {len(train_data)}")

## Prompt Definitions

5 prompts total:
- `prompt1`, `prompt2`, `prompt3`: three distinct well-crafted prompts (prompt2 uses CoT)
- `prompt_minimal`: almost no instructions (Problem 2)
- `prompt_icl`: in-context learning with 2 examples (Problem 3)

In [ ]:
def prompt1(ctx, endings):
    opts = "\n".join([f"{chr(65+i)}. {e}" for i, e in enumerate(endings)])
    return (
        f"Context: {ctx}\n\n"
        f"Which of the following best continues the context?\n"
        f"{opts}\n\n"
        f"Answer with a single letter (A, B, C, or D):\n"
    )

def prompt2(ctx, endings):  # Chain of Thought
    opts = "\n".join([f"{chr(65+i)}. {e}" for i, e in enumerate(endings)])
    return (
        f"Context: {ctx}\n\n"
        f"Options:\n{opts}\n\n"
        f"Think step by step about which option most naturally and logically continues the context. "
        f"Consider the subject, action, and setting. "
        f"After reasoning, give your final answer as a single letter (A, B, C, or D).\n"
    )

def prompt3(ctx, endings):
    opts = "\n".join([f"{i+1}. {e}" for i, e in enumerate(endings)])
    return (
        f"The following is a sentence completion task. "
        f"Read the context and pick the most plausible ending.\n\n"
        f"Context: {ctx}\n\n"
        f"{opts}\n\n"
        f"The correct ending number is:"
    )

def prompt_minimal(ctx, endings):
    opts = " ".join([f"{i+1}) {e}" for i, e in enumerate(endings)])
    return f"{ctx} {opts} Answer:"

# Two ICL examples drawn from train set
ICL_EXAMPLES = (
    "Context: A man is sitting on a roof. he\n"
    "A. is using wrap to wrap a pair of skis.\n"
    "B. is ripping level tiles off.\n"
    "C. is holding a rubik's cube.\n"
    "D. starts pulling up roofing on a roof.\n"
    "Answer: D\n\n"
    "Context: A lady walks to a barbell. She bends down and grabs the pole. the lady\n"
    "A. swings and lands in her arms.\n"
    "B. pulls the barbell forward.\n"
    "C. pulls a rope attached to the barbell.\n"
    "D. stands and lifts the weight over her head.\n"
    "Answer: D\n\n"
)

def prompt_icl(ctx, endings):
    opts = "\n".join([f"{chr(65+i)}. {e}" for i, e in enumerate(endings)])
    return (
        ICL_EXAMPLES +
        f"Context: {ctx}\n"
        f"{opts}\n"
        f"Answer:"
    )

PROMPTS = {
    "prompt1": prompt1,
    "prompt2_cot": prompt2,
    "prompt3": prompt3,
    "prompt_minimal": prompt_minimal,
    "prompt_icl": prompt_icl,
}

## Inference Helpers

In [ ]:
def parse_answer(text, prompt_name):
    text = text.strip()
    # look for first occurrence of A/B/C/D or 1/2/3/4
    for ch, idx in [("A",0),("B",1),("C",2),("D",3)]:
        if ch in text[:20]:
            return idx
    for ch, idx in [("1",0),("2",1),("3",2),("4",3)]:
        if ch in text[:20]:
            return idx
    return 0  # fallback


def run_causal_model(model_name, prompt_fn, data, max_new_tokens=50):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    model.eval()

    preds = []
    for item in tqdm(data, desc=model_name):
        text = prompt_fn(item["ctx"], item["endings"])
        inputs = tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        preds.append(parse_answer(generated, prompt_fn.__name__))

    del model
    torch.cuda.empty_cache()
    return preds


def run_seq2seq_model(model_name, prompt_fn, data, max_new_tokens=50):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    model.eval()

    preds = []
    for item in tqdm(data, desc=model_name):
        text = prompt_fn(item["ctx"], item["endings"])
        inputs = tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
            )
        generated = tokenizer.decode(out[0], skip_special_tokens=True)
        preds.append(parse_answer(generated, prompt_fn.__name__))

    del model
    torch.cuda.empty_cache()
    return preds


def run_mistral(prompt_fn, data, max_new_tokens=100):
    model_name = "mistralai/Mistral-7B-Instruct-v0.2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16, device_map="auto"
    )
    model.eval()

    preds = []
    for item in tqdm(data, desc="Mistral"):
        instruction = prompt_fn(item["ctx"], item["endings"])
        messages = [{"role": "user", "content": instruction}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        preds.append(parse_answer(generated, prompt_fn.__name__))

    del model
    torch.cuda.empty_cache()
    return preds


def accuracy(preds, data):
    labels = [item["label"] for item in data]
    correct = sum(p == l for p, l in zip(preds, labels))
    return correct / len(labels)

## Problem 5: Evaluate all prompts × all models

In [ ]:
results = {}  # results[model_name][prompt_name] = accuracy

# --- GPT2 ---
results["GPT2"] = {}
for pname, pfn in PROMPTS.items():
    preds = run_causal_model("openai-community/gpt2", pfn, val_data)
    results["GPT2"][pname] = accuracy(preds, val_data)
    print(f"GPT2 | {pname}: {results['GPT2'][pname]:.3f}")

In [ ]:
# --- Gemma-2b ---
results["Gemma-2b"] = {}
for pname, pfn in PROMPTS.items():
    preds = run_causal_model("google/gemma-2b", pfn, val_data)
    results["Gemma-2b"][pname] = accuracy(preds, val_data)
    print(f"Gemma-2b | {pname}: {results['Gemma-2b'][pname]:.3f}")

In [ ]:
# --- flan-t5-large ---
results["flan-t5-large"] = {}
for pname, pfn in PROMPTS.items():
    preds = run_seq2seq_model("google/flan-t5-large", pfn, val_data)
    results["flan-t5-large"][pname] = accuracy(preds, val_data)
    print(f"flan-t5-large | {pname}: {results['flan-t5-large'][pname]:.3f}")

In [ ]:
# --- OPT-2.7b ---
results["OPT-2.7b"] = {}
for pname, pfn in PROMPTS.items():
    preds = run_causal_model("facebook/opt-2.7b", pfn, val_data)
    results["OPT-2.7b"][pname] = accuracy(preds, val_data)
    print(f"OPT-2.7b | {pname}: {results['OPT-2.7b'][pname]:.3f}")

In [ ]:
# --- Mistral-7B-Instruct ---
results["Mistral-7B"] = {}
for pname, pfn in PROMPTS.items():
    preds = run_mistral(pfn, val_data)
    results["Mistral-7B"][pname] = accuracy(preds, val_data)
    print(f"Mistral-7B | {pname}: {results['Mistral-7B'][pname]:.3f}")

In [ ]:
# save results in case kernel dies
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)
print("saved")

## Problem 5: Plots

In [ ]:
# reload if needed
# with open("results.json") as f:
#     results = json.load(f)

prompt_names = list(PROMPTS.keys())
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]

fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 5), sharey=True)

for ax, (model_name, scores) in zip(axes, results.items()):
    accs = [scores[p] for p in prompt_names]
    bars = ax.bar(prompt_names, accs, color=colors)
    ax.set_title(model_name)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy")
    ax.set_xticks(range(len(prompt_names)))
    ax.set_xticklabels(prompt_names, rotation=30, ha="right", fontsize=8)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, acc + 0.01, f"{acc:.2f}",
                ha="center", va="bottom", fontsize=8)

plt.suptitle("Prompt Accuracy by Model", fontsize=14)
plt.tight_layout()
plt.savefig("results_plot.png", dpi=150, bbox_inches="tight")
plt.show()

## Problem 6: Generate test predictions with best prompt+model

In [ ]:
# find best combo
best_model, best_prompt, best_acc = None, None, 0
for model_name, scores in results.items():
    for pname, acc in scores.items():
        if acc > best_acc:
            best_acc = acc
            best_model = model_name
            best_prompt = pname

print(f"Best: {best_model} + {best_prompt} -> {best_acc:.3f}")

In [ ]:
MODEL_FN_MAP = {
    "GPT2": ("openai-community/gpt2", "causal"),
    "Gemma-2b": ("google/gemma-2b", "causal"),
    "flan-t5-large": ("google/flan-t5-large", "seq2seq"),
    "OPT-2.7b": ("facebook/opt-2.7b", "causal"),
    "Mistral-7B": (None, "mistral"),
}

pfn = PROMPTS[best_prompt]
model_id, mtype = MODEL_FN_MAP[best_model]

if mtype == "causal":
    test_preds = run_causal_model(model_id, pfn, test_data)
elif mtype == "seq2seq":
    test_preds = run_seq2seq_model(model_id, pfn, test_data)
else:
    test_preds = run_mistral(pfn, test_data)

print(f"Generated {len(test_preds)} predictions")

In [ ]:
submission = [{"ind": item["ind"], "label": pred} for item, pred in zip(test_data, test_preds)]
with open("submission.json", "w") as f:
    json.dump(submission, f)
print("submission.json saved")